In [5]:
# ----- Reproducibility: all random seeds documented here -----
SEED = 42                    # seed for RF and GBM
REPEATED_CV_SEEDS = range(20) # repeated-CV sweep (0..19) to quantify fold-split variance

import sys
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def prep(cols):
    """Scale continuous predictors; one-hot the categorical ones (never scaled)."""
    cat = [c for c in cols if c in CATEGORICAL]
    num = [c for c in cols if c not in CATEGORICAL]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(drop="if_binary", sparse_output=False), cat),
    ])


def _find_scripts_dir():
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for cand in (base / "scripts", base,
                     *sorted(base.glob("*/scripts")), *sorted(base.glob("*/*/scripts"))):
            if (cand / "project_paths.py").is_file():
                return cand
    raise FileNotFoundError(
        "scripts/project_paths.py not found. Open this notebook from inside the "
        "cloned repository, or point the kernel's working directory at it."
    )

sys.path.insert(0, str(_find_scripts_dir()))
from project_config import *   # ACOUSTIC, CLINICAL, DATA_DIR, repeated_cv, ...
import numpy as np
np.random.seed(SEED)  

# Modeling table + the feature sets the overview tables below iterate over.
# NOTE: these values are lists of COLUMN NAMES, because the cells below do
# model_df[cols]. The `feature_sets` in the "Clinical without GAD-7" cell is a
# different thing: it maps to DataFrames.
model_df = load_model_df()
y = model_df["phq9"]

feature_sets = {
    "Demographics":      DEMO,
    "Psych":             PSYCH,
    "Clinical":          CLINICAL,
    "Acoustic":          ACOUSTIC,
    "Demo+Acoustic":     DEMO_ACOUSTIC,
    "Clinical+Acoustic": CLINICAL_ACOUSTIC,
}
       # covers any incidental numpy randomness


In [ ]:
import warnings
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)

metrics = ["R2", "MAE", "r"]
print(f"{'feature set':18s}{'model':7s}" + "".join(f"{m:>16}" for m in metrics))
for sname, cols in feature_sets.items():
    models = {
        "Dummy": DummyRegressor(strategy="mean"),
        "Linear": Pipeline([("prep", prep(cols)),
                            ("est",  LinearRegression())]),
        "Poisson": Pipeline([("prep", prep(cols)),
                             ("est",  PoissonRegressor(alpha=1.0, max_iter=5000))]),
        "RF":    RandomForestRegressor(random_state=SEED),
        "GBM":   GradientBoostingRegressor(random_state=SEED),
    }
    for mname, m in models.items():
        res = repeated_cv(m, model_df[cols], y)
        cells = "".join(f"{f'{res[k][0]:.3f}±{res[k][1]:.3f}':>16}" for k in metrics)
        print(f"{sname:18s}{mname:7s}{cells}")


feature set       model                R2             MAE               r
Demographics      Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Demographics      Linear      0.356±0.024     5.281±0.112     0.602±0.018
Demographics      Poisson     0.291±0.031     5.777±0.142     0.547±0.023
Demographics      RF          0.188±0.061     5.964±0.254     0.495±0.039
Demographics      GBM         0.045±0.078     6.294±0.290     0.452±0.047
Psych             Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Psych             Linear      0.777±0.012     3.018±0.095     0.882±0.006
Psych             Poisson     0.467±0.130     4.096±0.220     0.751±0.029
Psych             RF          0.778±0.019     2.622±0.122     0.884±0.010
Psych             GBM         0.743±0.021     2.994±0.123     0.868±0.011
Clinical          Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Clinical          Linear      0.768±0.014     3.106±0.095     0.879±0.007
Clinical          Poisson     0.432±0.

In [ ]:
import warnings
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)

metrics = ["R2", "MAE", "r"]
print(f"{'feature set':18s}{'model':7s}" + "".join(f"{m:>16}" for m in metrics))
for sname, cols in feature_sets.items():
    models = {
        "Dummy": DummyRegressor(strategy="mean"),
        "Ridge": Pipeline([("prep", prep(cols)),
                           ("est",  RidgeCV(alphas=np.logspace(-2, 4, 25)))]),
        "RF":    RandomForestRegressor(random_state=SEED),
        "GBM":   GradientBoostingRegressor(random_state=SEED),
    }
    for mname, m in models.items():
        res = repeated_cv(m, model_df[cols], y)
        cells = "".join(f"{f'{res[k][0]:.3f}±{res[k][1]:.3f}':>16}" for k in metrics)
        print(f"{sname:18s}{mname:7s}{cells}")


feature set       model                R2             MAE               r
Demographics      Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Demographics      Ridge       0.356±0.022     5.372±0.113     0.597±0.018
Demographics      RF          0.188±0.061     5.964±0.254     0.495±0.039
Demographics      GBM         0.045±0.078     6.294±0.290     0.452±0.047
Psych             Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Psych             Ridge       0.777±0.011     3.066±0.091     0.882±0.006
Psych             RF          0.778±0.019     2.622±0.122     0.884±0.010
Psych             GBM         0.743±0.021     2.994±0.123     0.868±0.011
Clinical          Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
Clinical          Ridge       0.774±0.013     3.159±0.089     0.880±0.007
Clinical          RF          0.764±0.019     2.660±0.123     0.876±0.011
Clinical          GBM         0.700±0.024     2.988±0.114     0.845±0.012
Acoustic          Dummy      -0.044±0.

In [ ]:
# RF fit separately within each diagnostic group, every feature set.
# Same 20 x 10-fold repeated-CV protocol as the main table.
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# model_df has no `type` column, pull MDD/HC from the WAV inventory, keyed on int
# to bridge the zero-padded folder ids (same approach as error_anaylsis.ipynb).
inv = pd.read_csv(WAV_INVENTORY, dtype={"subject_id": str})
type_map = (inv.drop_duplicates("subject_id")
               .assign(k=lambda d: d["subject_id"].astype(int))
               .set_index("k")["type"])
groups = model_df["subject_key"].astype(int).map(type_map)
if groups.isna().any():
    print(f"warning: {groups.isna().sum()} subject(s) with no MDD/HC label excluded from subgroups\n")

GROUPS  = {"All": groups.notna(), "MDD": groups == "MDD", "HC": groups == "HC"}
rf      = RandomForestRegressor(random_state=SEED)
metrics = ["R2", "MAE", "r"]

rows = []
print(f"{'feature set':18s}{'group':6s}{'N':>4}{'p':>5}{'p/N':>6}{'PHQ9 sd':>9}"
      + "".join(f"{m:>16}" for m in metrics))
for sname, cols in feature_sets.items():
    for gname, mask in GROUPS.items():
        Xg, yg = model_df.loc[mask, cols], y[mask]
        n, p   = int(mask.sum()), len(cols)
        res    = repeated_cv(rf, Xg, yg)
        cells  = "".join(f"{f'{res[k][0]:.3f}±{res[k][1]:.3f}':>16}" for k in metrics)
        print(f"{sname:18s}{gname:6s}{n:>4}{p:>5}{p/n:>6.1f}{yg.std():>9.2f}{cells}")
        rows.append({"feature_set": sname, "group": gname, "n": n, "p": p,
                     "phq9_sd": round(yg.std(), 2),
                     **{f"{k}_{lab}": round(res[k][i], 3)
                        for k in metrics for i, lab in enumerate(["mean", "sd"])}})
    print()

subgroup = pd.DataFrame(rows)
subgroup.to_csv(DATA_DIR / "subgroup_results.csv", index=False)
print(f"Saved -> {DATA_DIR / 'subgroup_results.csv'}")


feature set       group    N    p   p/N  PHQ9 sd              R2             MAE               r
Demographics      All     52    3   0.1     8.48     0.188±0.061     5.964±0.254     0.495±0.039
Demographics      MDD     23    3   0.1     4.44    -0.379±0.144     4.032±0.214    -0.113±0.118
Demographics      HC      29    3   0.1     2.11    -0.168±0.184     1.626±0.100     0.244±0.115

Psych             All     52    4   0.1     8.48     0.778±0.019     2.622±0.122     0.884±0.010
Psych             MDD     23    4   0.2     4.44    -0.197±0.094     3.563±0.173     0.063±0.089
Psych             HC      29    4   0.1     2.11     0.122±0.044     1.506±0.059     0.408±0.042

Clinical          All     52    7   0.1     8.48     0.764±0.019     2.660±0.123     0.876±0.011
Clinical          MDD     23    7   0.3     4.44    -0.392±0.097     3.930±0.174    -0.111±0.093
Clinical          HC      29    7   0.2     2.11     0.042±0.049     1.507±0.055     0.325±0.040

Acoustic          All     5

In [ ]:
# Clinical-only baseline models (linear regression, Poisson regression)
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr


model_df = pd.read_csv(MODEL_DF_CSV)



# Clinical-only: select the clinical predictors explicitly (exclude all acoustic features).
clinical_predictors = ["age", "gender", "education_years", "ctq_sf", "LES", "SSRS", "gad7", "PSQI"]
X_clinical = model_df[clinical_predictors]
y = model_df["phq9"]

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)   # matches main pipeline (was n_splits=5)

lin = LinearRegression()
poisson = PoissonRegressor(alpha=0.0, max_iter=1000)

lin_pred     = cross_val_predict(lin, X_clinical, y, cv=kf, n_jobs=1)
poisson_pred = cross_val_predict(poisson, X_clinical, y, cv=kf, n_jobs=1)

def report(name, pred):
    print(f"{name:18s} MAE: {mean_absolute_error(y, pred):.3f}  "
          f"RMSE: {np.sqrt(mean_squared_error(y, pred)):.3f}  "
          f"R2: {r2_score(y, pred):.3f}  "
          f"r: {pearsonr(y, pred)[0]:.3f}")

report("Linear Regression", lin_pred)
report("Poisson Regression", poisson_pred)

Linear Regression  MAE: 3.144  RMSE: 3.994  R2: 0.774  r: 0.882
Poisson Regression MAE: 4.584  RMSE: 8.409  R2: -0.002  r: 0.662


In [ ]:
# GAD-7 only -> PHQ-9
import warnings
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)

models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25))),
    "RF":    RandomForestRegressor(random_state=SEED),
    "GBM":   GradientBoostingRegressor(random_state=SEED),
}

gad7_only = {"GAD-7 only": ["gad7"]}      # list of column names -> model_df[cols]

metrics = ["R2", "MAE", "r"]
print(f"{'feature set':18s}{'model':7s}" + "".join(f"{m:>16}" for m in metrics))
for sname, cols in gad7_only.items():
    for mname, m in models.items():
        res = repeated_cv(m, model_df[cols], y)
        cells = "".join(f"{f'{res[k][0]:.3f}±{res[k][1]:.3f}':>16}" for k in metrics)
        print(f"{sname:18s}{mname:7s}{cells}")


feature set       model                R2             MAE               r
GAD-7 only        Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
GAD-7 only        Ridge       0.775±0.007     3.071±0.053     0.881±0.004
GAD-7 only        RF          0.721±0.024     3.027±0.123     0.855±0.012
GAD-7 only        GBM         0.679±0.032     3.253±0.132     0.835±0.015


In [ ]:
# PSQI only -> PHQ-9
import warnings
import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings("ignore", category=ConvergenceWarning)

models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25))),
    "RF":    RandomForestRegressor(random_state=SEED),
    "GBM":   GradientBoostingRegressor(random_state=SEED),
}

psqi_only = {"PSQI only": ["PSQI"]} 

metrics = ["R2", "MAE", "r"]
print(f"{'feature set':18s}{'model':7s}" + "".join(f"{m:>16}" for m in metrics))
for sname, cols in psqi_only.items():
    for mname, m in models.items():
        res = repeated_cv(m, model_df[cols], y)
        cells = "".join(f"{f'{res[k][0]:.3f}±{res[k][1]:.3f}':>16}" for k in metrics)
        print(f"{sname:18s}{mname:7s}{cells}")


feature set       model                R2             MAE               r
PSQI only         Dummy      -0.044±0.022     7.980±0.086    -0.419±0.104
PSQI only         Ridge       0.595±0.009     4.201±0.063     0.771±0.006
PSQI only         RF          0.637±0.040     3.486±0.169     0.806±0.022
PSQI only         GBM         0.614±0.035     3.668±0.144     0.797±0.018


In [ ]:
%pip install --upgrade "scikit-learn>=1.5" scipy


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Permutation test score to test statistical significance of the models (10-fold CV, 1000 permutations)

from sklearn.model_selection import permutation_test_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import pandas as pd

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
models = {"RF": RandomForestRegressor(random_state=SEED),
          "GBM": GradientBoostingRegressor(random_state=SEED)}
feature_sets = {
    "Demographics": DEMO, "Psych": PSYCH, "Clinical": CLINICAL,
    "Acoustic": ACOUSTIC, "Demo+Acoustic": DEMO_ACOUSTIC, "Clinical+Acoustic": CLINICAL_ACOUSTIC,
}

rows = []
for sname, cols in feature_sets.items():
    for mname, m in models.items():
        score, _, pval = permutation_test_score(
            m, model_df[cols], y, cv=kf,
            scoring="neg_mean_absolute_error",     # stable statistic; flip sign below
            n_permutations=1000, random_state=SEED, n_jobs=-1)
        rows.append({"feature_set": sname, "model": mname,
                     "cv_MAE": round(-score, 3), "p_value": round(pval, 4)})

perm = pd.DataFrame(rows)
print(perm.to_string(index=False))
perm.to_csv(DATA_DIR / "permutation_test_results.csv", index=False)


      feature_set model  cv_MAE  p_value
     Demographics    RF   5.911    0.004
     Demographics   GBM   6.144    0.003
            Psych    RF   2.680    0.001
            Psych   GBM   3.057    0.001
         Clinical    RF   2.705    0.001
         Clinical   GBM   2.907    0.001
         Acoustic    RF   6.160    0.002
         Acoustic   GBM   5.624    0.002
    Demo+Acoustic    RF   5.988    0.002
    Demo+Acoustic   GBM   6.039    0.007
Clinical+Acoustic    RF   2.877    0.001
Clinical+Acoustic   GBM   3.026    0.001


In [ ]:
# Version check for key packages (numpy, pandas, scikit-learn, scipy, matplotlib, librosa, soundfile, shap)
from importlib.metadata import version, PackageNotFoundError
for pkg in ["numpy", "pandas", "scikit-learn", "scipy",
            "matplotlib", "librosa", "soundfile", "shap"]:
    try:
        print(f"{pkg:15s}{version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:15s}(not installed)")

import sys
print(f"{'Python':15s}{sys.version.split()[0]}")



numpy          2.2.6
pandas         2.2.2
scikit-learn   1.9.0
scipy          1.17.1
matplotlib     3.8.4
librosa        0.11.0
soundfile      0.14.0
shap           0.51.0
Python         3.11.3
